In [1]:
# Core libraries
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Machine-learning tools
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_score,
    learning_curve,
)
from sklearn.preprocessing import (
    LabelEncoder,
    label_binarize,
)

In [2]:
# Assemble the Cluster 2 single-AA dataset.

# DATA_DIR = Path(".")
DATA_DIR = Path(
    "/Volumes/JAEONCHO/ADIC-AA/g_CD data"
)

TRIM_EVENT_DIR = (
    DATA_DIR / "trim_events_g_CD"
)

TRIM_SUB_EVENT_DIR = (
    DATA_DIR / "trim_sub_events_g_CD"
)

BASELINE_DIR = (
    DATA_DIR / "final_baseline_info_g_CD"
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CLUSTER_ID = 2

CLUSTER_AA_CLASSES = [
    "L",
    "M",
]

assembled_datasets = []

for sub_event_file in sorted(
    TRIM_SUB_EVENT_DIR.glob("*.csv")
):
    recording_id = (
        sub_event_file.stem.removesuffix("_sub")
    )

    aa_label = recording_id[0]

    if aa_label not in CLUSTER_AA_CLASSES:
        continue

    event_file = (
        TRIM_EVENT_DIR
        / f"{recording_id}.csv"
    )

    baseline_file = (
        BASELINE_DIR
        / f"{recording_id}_final.csv"
    )

    if not event_file.exists():
        raise FileNotFoundError(
            f"Missing trimmed event file: "
            f"{event_file.name}"
        )

    if not baseline_file.exists():
        raise FileNotFoundError(
            f"Missing baseline file: "
            f"{baseline_file.name}"
        )

    sub_events = pd.read_csv(
        sub_event_file
    )

    first_level_events = pd.read_csv(
        event_file
    )

    baseline_data = pd.read_csv(
        baseline_file
    )

    if baseline_data.empty:
        raise ValueError(
            f"Baseline file is empty: "
            f"{baseline_file.name}"
        )

    open_pore_rms = (
        baseline_data["rms"].iloc[0]
    )

    matched_rows = []

    for _, sub_event in sub_events.iterrows():
        parent_event = first_level_events.loc[
            (
                first_level_events["start_time_s"]
                <= sub_event["sub_start_time_s"]
            )
            & (
                first_level_events["end_time_s"]
                >= sub_event["sub_end_time_s"]
            )
        ]

        if parent_event.empty:
            continue

        parent_event = parent_event.iloc[0]

        event_id = (
            f"{recording_id}_"
            f"{int(parent_event['event_index'])}"
        )

        matched_rows.append({
            "sub_pre_interaction_current":
                sub_event[
                    "sub_pre_interaction_current"
                ],

            "sub_post_interaction_current":
                sub_event[
                    "sub_post_interaction_current"
                ],

            # I1
            "first_level_current":
                sub_event["sub_mode_baseline"],

            # σ1
            "first_level_stdev":
                parent_event["blockage_stdev"],

            # σ0
            "open_pore_rms":
                open_pore_rms,

            "dwell_time_us":
                parent_event["dwell_time_us"],

            # Parent first-level blockage
            "event_id":
                event_id,

            # Single-pore recording
            "dataset_id":
                recording_id,

            # One-letter AA label required by the
            # downstream two-level classifier.
            "AA":
                aa_label,
        })

    if matched_rows:
        assembled_datasets.append(
            pd.DataFrame(matched_rows)
        )

if not assembled_datasets:
    raise ValueError(
        "No matched Cluster 2 observations were found."
    )

df = pd.concat(
    assembled_datasets,
    ignore_index=True,
)

required_columns = [
    "sub_pre_interaction_current",
    "sub_post_interaction_current",
    "first_level_current",
    "first_level_stdev",
    "open_pore_rms",
    "dwell_time_us",
    "event_id",
    "dataset_id",
    "AA",
]

if df[required_columns].isna().any().any():
    missing_columns = (
        df[required_columns]
        .columns[
            df[required_columns]
            .isna()
            .any()
        ]
        .tolist()
    )

    raise ValueError(
        f"Missing values detected in: "
        f"{missing_columns}"
    )

if (df["open_pore_rms"] <= 0).any():
    raise ValueError(
        "All open-pore RMS values must be positive."
    )

observed_classes = set(
    df["AA"].unique()
)

expected_classes = set(
    CLUSTER_AA_CLASSES
)

if observed_classes != expected_classes:
    raise ValueError(
        "Cluster 2 class mismatch. "
        f"Expected {sorted(expected_classes)}, "
        f"found {sorted(observed_classes)}."
    )

print(
    f"Cluster {CLUSTER_ID} classes: "
    f"{sorted(observed_classes)}"
)

print(
    f"Secondary events: "
    f"{len(df):,}"
)

print(
    f"First-level events: "
    f"{df['event_id'].nunique():,}"
)

print(
    f"Single-pore recordings: "
    f"{df['dataset_id'].nunique():,}"
)

display(df.head())

Cluster 2 classes: ['L', 'M']
Secondary events: 11,151
First-level events: 653
Single-pore recordings: 11


,sub_pre_interaction_current,sub_post_interaction_current,first_level_current,first_level_stdev,open_pore_rms,dwell_time_us,event_id,dataset_id,AA
0,113.830560,118.25561,129.60814,26.281168,2.322577,56530.0,L_g_CD_250501__0001_0,L_g_CD_250501__0001,L
1,105.529780,117.02473,129.60814,26.281168,2.322577,56530.0,L_g_CD_250501__0001_0,L_g_CD_250501__0001,L
2,119.186390,116.97387,129.60814,26.281168,2.322577,56530.0,L_g_CD_250501__0001_0,L_g_CD_250501__0001,L
3,105.194084,115.72265,129.60814,26.281168,2.322577,56530.0,L_g_CD_250501__0001_0,L_g_CD_250501__0001,L
4,109.954830,117.14680,129.60814,26.281168,2.322577,56530.0,L_g_CD_250501__0001_0,L_g_CD_250501__0001,L


In [3]:
# Encode AA labels, split by parent first-level blockage,
# and construct the four classifier features.

FEATURE_NAMES = [
    "Baseline Ratio",
    "Z Score",
    "Per-event Impact",
    "Per-time Impact",
]

# Encode one-letter AA labels.
label_encoder = LabelEncoder()
label_encoder.fit(df["AA"])

label_mapping = {
    class_name: int(encoded_label)
    for class_name, encoded_label
    in zip(
        label_encoder.classes_,
        label_encoder.transform(
            label_encoder.classes_
        ),
    )
}

LABEL_MAPPING_PATH = (
    OUTPUT_DIR
    / f"cluster{CLUSTER_ID}_label_mapping.joblib"
)

joblib.dump(
    label_mapping,
    LABEL_MAPPING_PATH,
)

print(
    "Label mapping:",
    label_mapping,
)

# Each event_id represents one parent first-level blockage.
event_table = (
    df[
        [
            "event_id",
            "dataset_id",
            "AA",
        ]
    ]
    .drop_duplicates(subset="event_id")
    .sort_values(["AA", "event_id"])
    .reset_index(drop=True)
)

event_validation = (
    df.groupby("event_id")
    .agg(
        AA_count=("AA", "nunique"),
        dataset_count=("dataset_id", "nunique"),
    )
)

if (
    (event_validation["AA_count"] != 1).any()
    or (
        event_validation["dataset_count"] != 1
    ).any()
):
    raise ValueError(
        "Each event_id must correspond to one AA class "
        "and one single-pore recording."
    )

# Deterministic 60:20:20 target split within each AA.
rng = np.random.default_rng(42)
split_assignment = {}

for aa_name in sorted(
    event_table["AA"].unique()
):
    aa_event_ids = (
        event_table.loc[
            event_table["AA"] == aa_name,
            "event_id",
        ]
        .sort_values()
        .to_numpy()
    )

    aa_event_ids = rng.permutation(
        aa_event_ids
    )

    number_of_events = len(
        aa_event_ids
    )

    number_train = max(
        1,
        int(
            np.floor(
                0.60 * number_of_events
            )
        ),
    )

    number_validation = max(
        1,
        int(
            np.floor(
                0.20 * number_of_events
            )
        ),
    )

    if (
        number_train
        + number_validation
        >= number_of_events
    ):
        number_validation = max(
            0,
            number_of_events
            - number_train
            - 1,
        )

    train_event_ids = (
        aa_event_ids[:number_train]
    )

    validation_event_ids = (
        aa_event_ids[
            number_train:
            number_train
            + number_validation
        ]
    )

    test_event_ids = (
        aa_event_ids[
            number_train
            + number_validation:
        ]
    )

    for event_id in train_event_ids:
        split_assignment[event_id] = "train"

    for event_id in validation_event_ids:
        split_assignment[event_id] = "validation"

    for event_id in test_event_ids:
        split_assignment[event_id] = "test"

# Ensure that every recording contributes training data
# for calculation of the recording-level features.
for dataset_id in sorted(
    event_table["dataset_id"].unique()
):
    recording_event_ids = (
        event_table.loc[
            event_table["dataset_id"]
            == dataset_id,
            "event_id",
        ]
        .sort_values()
        .tolist()
    )

    has_training_event = any(
        split_assignment[event_id] == "train"
        for event_id in recording_event_ids
    )

    if not has_training_event:
        event_to_move = (
            recording_event_ids[0]
        )

        split_assignment[
            event_to_move
        ] = "train"

df["split"] = df["event_id"].map(
    split_assignment
)

if df["split"].isna().any():
    raise ValueError(
        "Some observations were not assigned to a split."
    )

train_mask = df["split"] == "train"
validation_mask = (
    df["split"] == "validation"
)
test_mask = df["split"] == "test"

# Calculate recording-level aggregate values using
# training observations only.
training_rows = df.loc[
    train_mask
].copy()

training_event_count = (
    training_rows
    .groupby("dataset_id")
    .size()
    .rename(
        "training_secondary_event_count"
    )
)

training_dwell_time = (
    training_rows[
        [
            "dataset_id",
            "event_id",
            "dwell_time_us",
        ]
    ]
    .drop_duplicates(subset="event_id")
    .groupby("dataset_id")[
        "dwell_time_us"
    ]
    .sum()
    .rename(
        "training_first_level_dwell_time"
    )
)

training_aggregates = pd.concat(
    [
        training_event_count,
        training_dwell_time,
    ],
    axis=1,
)

training_aggregates[
    "training_event_frequency"
] = (
    training_aggregates[
        "training_secondary_event_count"
    ]
    / training_aggregates[
        "training_first_level_dwell_time"
    ]
)

if (
    training_aggregates.isna().any().any()
    or (
        training_aggregates[
            "training_secondary_event_count"
        ]
        <= 0
    ).any()
    or (
        training_aggregates[
            "training_first_level_dwell_time"
        ]
        <= 0
    ).any()
    or (
        training_aggregates[
            "training_event_frequency"
        ]
        <= 0
    ).any()
):
    raise ValueError(
        "Invalid training-derived aggregate values."
    )

feature_source = df.copy()

feature_source[
    "training_secondary_event_count"
] = feature_source["dataset_id"].map(
    training_aggregates[
        "training_secondary_event_count"
    ]
)

feature_source[
    "training_first_level_dwell_time"
] = feature_source["dataset_id"].map(
    training_aggregates[
        "training_first_level_dwell_time"
    ]
)

feature_source[
    "training_event_frequency"
] = feature_source["dataset_id"].map(
    training_aggregates[
        "training_event_frequency"
    ]
)

aggregate_columns = [
    "training_secondary_event_count",
    "training_first_level_dwell_time",
    "training_event_frequency",
]

if (
    feature_source[
        aggregate_columns
    ]
    .isna()
    .any()
    .any()
):
    missing_recordings = (
        feature_source.loc[
            feature_source[
                aggregate_columns
            ]
            .isna()
            .any(axis=1),
            "dataset_id",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "No training-derived aggregate values for: "
        f"{missing_recordings}"
    )

# Construct the four model features.
feature_source["Baseline Ratio"] = (
    feature_source[
        "sub_post_interaction_current"
    ]
    / feature_source[
        "first_level_current"
    ]
)

feature_source["Z Score"] = (
    (
        feature_source[
            "sub_post_interaction_current"
        ]
        - feature_source[
            "first_level_current"
        ]
    )
    / feature_source[
        "open_pore_rms"
    ]
)

feature_source["Per-event Impact"] = (
    feature_source["Z Score"] ** 2
    / feature_source[
        "training_secondary_event_count"
    ]
)

feature_source["Per-time Impact"] = (
    feature_source["Z Score"] ** 2
    / feature_source[
        "training_event_frequency"
    ]
)

feature_values = (
    feature_source[
        FEATURE_NAMES
    ]
    .to_numpy()
)

if not np.isfinite(feature_values).all():
    raise ValueError(
        "Nonfinite values detected in model features."
    )

X_train = (
    feature_source.loc[
        train_mask,
        FEATURE_NAMES,
    ]
    .copy()
    .reset_index(drop=True)
)

X_val = (
    feature_source.loc[
        validation_mask,
        FEATURE_NAMES,
    ]
    .copy()
    .reset_index(drop=True)
)

X_test = (
    feature_source.loc[
        test_mask,
        FEATURE_NAMES,
    ]
    .copy()
    .reset_index(drop=True)
)

y_train = label_encoder.transform(
    feature_source.loc[
        train_mask,
        "AA",
    ]
)

y_val = label_encoder.transform(
    feature_source.loc[
        validation_mask,
        "AA",
    ]
)

y_test = label_encoder.transform(
    feature_source.loc[
        test_mask,
        "AA",
    ]
)

groups_train = (
    feature_source.loc[
        train_mask,
        "event_id",
    ]
    .reset_index(drop=True)
    .to_numpy()
)

split_assignment_df = (
    feature_source[
        [
            "event_id",
            "dataset_id",
            "AA",
            "split",
        ]
    ]
    .drop_duplicates(subset="event_id")
    .sort_values(["AA", "event_id"])
    .reset_index(drop=True)
)

split_assignment_df.to_csv(
    OUTPUT_DIR
    / f"cluster{CLUSTER_ID}_split_assignment.csv",
    index=False,
)

print(
    f"Training observations: "
    f"{len(X_train):,}"
)

print(
    f"Validation observations: "
    f"{len(X_val):,}"
)

print(
    f"Test observations: "
    f"{len(X_test):,}"
)

print(
    f"Label mapping saved to: "
    f"{LABEL_MAPPING_PATH.resolve()}"
)

Label mapping: {'L': 0, 'M': 1}
Training observations: 6,781
Validation observations: 2,136
Test observations: 2,234
Label mapping saved to: /Users/jaeoncho/Desktop/lab/gcd_aa/codes/jupyter/ML_gCD/outputs/cluster2_label_mapping.joblib


In [4]:
# Train and save the Cluster 2 single-AA classifier.

classifier = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1,
)

classifier.fit(
    X_train,
    y_train,
)

MODEL_PATH = (
    OUTPUT_DIR
    / f"cluster{CLUSTER_ID}_single_AA_classifier.joblib"
)

joblib.dump(
    classifier,
    MODEL_PATH,
)

print(
    f"Model saved to: "
    f"{MODEL_PATH.resolve()}"
)

Model saved to: /Users/jaeoncho/Desktop/lab/gcd_aa/codes/jupyter/ML_gCD/outputs/cluster2_single_AA_classifier.joblib


In [5]:
# Evaluate the Cluster 2 classifier.

grouped_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_scores = cross_val_score(
    clone(classifier),
    X_train,
    y_train,
    groups=groups_train,
    cv=grouped_cv,
    scoring="accuracy",
    n_jobs=-1,
)

y_test_pred = classifier.predict(
    X_test
)

y_test_proba = classifier.predict_proba(
    X_test
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred,
)

validation_accuracy = classifier.score(
    X_val,
    y_val,
)

f1_micro = f1_score(
    y_test,
    y_test_pred,
    average="micro",
)

f1_macro = f1_score(
    y_test,
    y_test_pred,
    average="macro",
)

precision_micro = precision_score(
    y_test,
    y_test_pred,
    average="micro",
    zero_division=0,
)

recall_micro = recall_score(
    y_test,
    y_test_pred,
    average="micro",
    zero_division=0,
)

# Binary classifiers require a one-dimensional
# probability array for the positive class.
if len(classifier.classes_) == 2:
    binary_auc = roc_auc_score(
        y_test,
        y_test_proba[:, 1],
    )

    auc_macro = binary_auc
    auc_micro = binary_auc

# Multiclass classifiers use the full
# class-probability matrix.
else:
    auc_macro = roc_auc_score(
        y_test,
        y_test_proba,
        multi_class="ovr",
        average="macro",
    )

    auc_micro = roc_auc_score(
        y_test,
        y_test_proba,
        multi_class="ovr",
        average="micro",
    )

metrics_summary = pd.DataFrame({
    "metric": [
        "test_accuracy",
        "validation_accuracy",
        "cross_validation_mean",
        "cross_validation_std",
        "f1_micro",
        "f1_macro",
        "precision_micro",
        "recall_micro",
        "auc_macro",
        "auc_micro",
    ],
    "value": [
        test_accuracy,
        validation_accuracy,
        cv_scores.mean(),
        cv_scores.std(),
        f1_micro,
        f1_macro,
        precision_micro,
        recall_micro,
        auc_macro,
        auc_micro,
    ],
})

cv_results = pd.DataFrame({
    "fold": np.arange(
        1,
        len(cv_scores) + 1,
    ),
    "accuracy": cv_scores,
})

metrics_summary.to_csv(
    OUTPUT_DIR
    / f"cluster{CLUSTER_ID}_metrics.csv",
    index=False,
)

cv_results.to_csv(
    OUTPUT_DIR
    / f"cluster{CLUSTER_ID}_cross_validation.csv",
    index=False,
)

display(metrics_summary)
display(cv_results)

,metric,value
0,test_accuracy,0.997762
1,validation_accuracy,0.999064
2,cross_validation_mean,0.997792
3,cross_validation_std,0.002153
4,f1_micro,0.997762
5,f1_macro,0.997757
6,precision_micro,0.997762
7,recall_micro,0.997762
8,auc_macro,0.999969
9,auc_micro,0.999969


,fold,accuracy
0,1,1.000000
1,2,0.997857
2,3,1.000000
3,4,0.994253
4,5,0.996850
